# Getting Started with ARBS - A Researcher's Guide

Welcome! This notebook will guide you through running your first backtest using the ARBS (Algorithmic Relative-value Bond Strategies) framework.

**No programming experience required** - just follow along and run each cell!

## What You'll Learn

1. How to load a pre-built trading strategy
2. How to run a backtest
3. How to understand the results
4. How to modify basic parameters

## Running This Notebook

- Press `Shift + Enter` to run each cell
- Run cells in order from top to bottom
- Don't worry if you see warnings - they're usually harmless

---

## Step 1: Setup

First, we'll import the tools we need. Think of this as opening your toolbox.

In [ ]:
# Add parent directory to path so we can import ARBS modules
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Import the strategy creation tools
from Strategies.Registry import quick_strategy, list_templates

# Import tools for running backtests
import numpy as np
import pandas as pd
from datetime import date

# Import plotting tools for visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots look nice
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Setup complete!")

---

## Step 2: See What Strategies Are Available

ARBS comes with several pre-built strategy templates. Let's see what's available:

In [ ]:
# List all available strategy templates
templates = list_templates()

print("Available Strategy Templates:")
print("=" * 40)
for template in templates:
    print(f"  • {template}")
print()
print(f"Total: {len(templates)} templates available")

---

## Step 3: Create Your First Strategy

Let's create a simple **carry strategy**. This strategy:
- Looks at interest rate futures contracts
- Identifies which contracts have higher "carry" (expected return from holding)
- Buys high-carry contracts and sells low-carry contracts

We'll start with just 3 contracts to keep it simple.

In [ ]:
# Define which futures contracts to trade
# These are SOFR (Secured Overnight Financing Rate) futures
# - SFRZ4 = December 2024
# - SFRH5 = March 2025
# - SFRM5 = June 2025
instruments = ['SFRZ4', 'SFRH5', 'SFRM5']

# Create the strategy using the 'simple_carry' template
strategy = quick_strategy('simple_carry', instruments=instruments)

print("✓ Strategy created!")
print()
print(f"Strategy name: {strategy.identifier}")
print(f"Number of signals: {len(strategy.signals)}")
print(f"Risk aversion: {strategy.optimizer.risk_aversion}")

---

## Step 4: Understanding the Strategy Components

Every strategy has four main components:

1. **Signals** - What to trade (e.g., "buy high carry contracts")
2. **Alpha Generator** - How strong we think the signal is
3. **Risk Model** - How risky each position is
4. **Optimizer** - How much to allocate to each position

Let's inspect our strategy:

In [ ]:
print("Strategy Components:")
print("=" * 50)
print()

print("1. SIGNALS")
for i, signal in enumerate(strategy.signals, 1):
    print(f"   Signal {i}: {signal.name}")
print()

print("2. ALPHA GENERATOR")
print(f"   IC (Information Coefficient): {strategy.alpha_generator.IC}")
print(f"   This represents expected correlation between signal and returns")
print(f"   Higher IC = stronger signal (range: 0.0 to 0.20 is typical)")
print()

print("3. RISK MODEL")
print(f"   Type: Ledoit-Wolf shrinkage covariance")
print(f"   This estimates how positions move together")
print()

print("4. OPTIMIZER")
print(f"   Risk aversion: {strategy.optimizer.risk_aversion}")
print(f"   Higher risk aversion = more conservative positions")
print(f"   Long only: {strategy.optimizer.long_only}")

---

## Step 5: Run a Backtest

Now let's test how this strategy would have performed historically.

**Important Note**: This uses synthetic data for demonstration purposes. In production, you would connect to real market data.

In [ ]:
# For this example, we'll create synthetic market data
# In production, you would connect to your market data provider

class SimpleMockMDP:
    """Mock market data provider for demonstration."""
    
    def __init__(self, base_rate=5.0, carry_spread=0.10):
        self.base_rate = base_rate
        self.carry_spread = carry_spread
    
    def get_pricer(self, currency, as_of):
        return self
    
    def futures_price(self, contract):
        # Simple pricing model with carry structure
        quarter_map = {'H': 0, 'M': 1, 'U': 2, 'Z': 3}
        quarter_code = contract[-2] if len(contract) >= 2 else 'H'
        quarter = quarter_map.get(quarter_code, 0)
        
        rate = self.base_rate + (quarter * self.carry_spread)
        price = 100.0 - rate
        noise = np.random.normal(0, 0.01)
        
        return price + noise

# Create mock data provider
mdp = SimpleMockMDP(base_rate=5.0, carry_spread=0.10)

print("✓ Mock market data created")
print()
print("This simulates a market with:")
print("  • Base rate: 5.0%")
print("  • Carry spread: 10 basis points per quarter")
print("  • Small random noise to simulate market movements")

In [ ]:
# Import backtest engine
from Backtest.Backtest import Backtest

# Set up the backtest
backtest = Backtest(
    mdp=mdp,
    risk_aversion=1.0,
    long_only=True,
    min_history=5
)

# Define the backtest period (3 months, weekly rebalancing)
start_date = date(2024, 9, 1)
end_date = date(2024, 12, 1)
dates = pd.date_range(start_date, end_date, freq='W').tolist()
dates = [d.date() if hasattr(d, 'date') else d for d in dates]

print(f"Running backtest from {start_date} to {end_date}...")
print(f"Rebalancing: Weekly ({len(dates)} periods)")
print()

# Set random seed for reproducibility
np.random.seed(42)

# Run the backtest!
result = backtest.run(contracts=instruments, dates=dates)

print("✓ Backtest complete!")

---

## Step 6: Understanding the Results

Let's look at the key performance metrics:

In [ ]:
print("Performance Metrics")
print("=" * 50)
print()

print(f"📊 Sharpe Ratio: {result.sharpe_ratio:.3f}")
print(f"   What it means: Risk-adjusted return")
print(f"   • Above 1.0 = Good")
print(f"   • Above 2.0 = Excellent")
print(f"   • Below 0.0 = Lost money relative to risk taken")
print()

print(f"🎯 Information Coefficient (IC): {result.ic:.3f}")
print(f"   What it means: How well the signal predicted returns")
print(f"   • 0.05 to 0.10 = Typical for good strategies")
print(f"   • Above 0.10 = Very strong signal")
print(f"   • Negative = Signal was backwards")
print()

print(f"💰 Total Return: {result.total_return:.2%}")
print(f"   What it means: How much money you made/lost")
print(f"   • This is over the {(end_date - start_date).days} day period")
print()

print(f"📈 Statistics:")
print(f"   Number of periods: {len(result.returns)}")
print(f"   Mean weekly return: {result.returns.mean():.4%}")
print(f"   Std deviation: {result.returns.std():.4%}")

---

## Step 7: Visualize the Results

Let's create some charts to better understand what happened:

In [ ]:
# Create a figure with 3 subplots
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

# 1. Cumulative Returns
cumulative_returns = (1 + result.returns).cumprod()
axes[0].plot(cumulative_returns.index, cumulative_returns.values, linewidth=2, color='steelblue')
axes[0].axhline(y=1.0, color='black', linestyle='--', alpha=0.3)
axes[0].set_title('Cumulative Returns (Growth of $1)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Portfolio Value', fontsize=12)
axes[0].grid(True, alpha=0.3)

# 2. Weekly Returns
colors = ['green' if r > 0 else 'red' for r in result.returns]
axes[1].bar(result.returns.index, result.returns.values, color=colors, alpha=0.6)
axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[1].set_title('Weekly Returns', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Return', fontsize=12)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.1%}'.format(y)))
axes[1].grid(True, alpha=0.3)

# 3. Portfolio Weights (average over time)
avg_weights = result.weights.mean()
axes[2].barh(avg_weights.index, avg_weights.values, color='steelblue', alpha=0.7)
axes[2].set_title('Average Portfolio Weights', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Weight', fontsize=12)
axes[2].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '{:.1%}'.format(x)))
axes[2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("✓ Visualizations complete!")

---

## Step 8: Examine Portfolio Weights

Let's see how the strategy allocated capital across contracts:

In [ ]:
print("Portfolio Allocation Summary")
print("=" * 50)
print()

# Show first period
print("First Period Weights:")
first_weights = result.weights.iloc[0]
for contract, weight in first_weights.items():
    if abs(weight) > 0.001:
        bar = '█' * int(weight * 50)
        print(f"  {contract}: {weight:>6.2%} {bar}")
print()

# Show last period
if len(result.weights) > 1:
    print("Last Period Weights:")
    last_weights = result.weights.iloc[-1]
    for contract, weight in last_weights.items():
        if abs(weight) > 0.001:
            bar = '█' * int(weight * 50)
            print(f"  {contract}: {weight:>6.2%} {bar}")
    print()

# Show average weights
print("Average Weights Over Time:")
avg_weights = result.weights.mean()
for contract, weight in avg_weights.items():
    if abs(weight) > 0.001:
        bar = '█' * int(weight * 50)
        print(f"  {contract}: {weight:>6.2%} {bar}")

---

## Step 9: Modify Strategy Parameters

Now let's try adjusting some parameters to see how they affect performance.

We'll test with **different risk aversion** levels:

In [ ]:
# Test different risk aversion levels
risk_aversions = [0.5, 1.0, 2.0, 3.0]
results = {}

print("Testing different risk aversion levels...")
print()

for ra in risk_aversions:
    # Create backtest with different risk aversion
    backtest = Backtest(
        mdp=mdp,
        risk_aversion=ra,
        long_only=True,
        min_history=5
    )
    
    # Run backtest
    np.random.seed(42)  # Same seed for fair comparison
    result = backtest.run(contracts=instruments, dates=dates)
    results[ra] = result
    
    print(f"Risk Aversion = {ra}:")
    print(f"  Sharpe: {result.sharpe_ratio:>6.3f}")
    print(f"  Return: {result.total_return:>6.2%}")
    print()

print("✓ Sensitivity analysis complete!")

In [ ]:
# Visualize the impact of risk aversion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Extract metrics
sharpes = [results[ra].sharpe_ratio for ra in risk_aversions]
returns = [results[ra].total_return for ra in risk_aversions]

# Plot Sharpe ratios
axes[0].plot(risk_aversions, sharpes, marker='o', linewidth=2, markersize=8, color='steelblue')
axes[0].set_title('Sharpe Ratio vs Risk Aversion', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Risk Aversion', fontsize=12)
axes[0].set_ylabel('Sharpe Ratio', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.3)

# Plot total returns
axes[1].plot(risk_aversions, returns, marker='s', linewidth=2, markersize=8, color='green')
axes[1].set_title('Total Return vs Risk Aversion', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Risk Aversion', fontsize=12)
axes[1].set_ylabel('Total Return', fontsize=12)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: '{:.1%}'.format(y)))
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Key Insight:")
print("Higher risk aversion = smaller positions = lower returns but also lower risk")
print("The optimal risk aversion depends on your risk tolerance!")

---

## Step 10: Try a Different Strategy

Let's compare our carry strategy to a momentum strategy:

In [ ]:
# Create a momentum strategy
momentum_strategy = quick_strategy(
    'multi_signal',  # This template includes momentum
    instruments=instruments
)

print("✓ Momentum strategy created!")
print()
print(f"Strategy: {momentum_strategy.identifier}")
print(f"Signals: {[s.name for s in momentum_strategy.signals]}")
print()
print("This strategy:")
print("  • Looks at recent price trends")
print("  • Buys contracts that have been going up")
print("  • Sells contracts that have been going down")

---

## Summary: What You've Learned

Congratulations! You've now:

✅ Created a trading strategy using templates

✅ Run a backtest to evaluate performance

✅ Interpreted key metrics (Sharpe, IC, returns)

✅ Visualized portfolio performance

✅ Tested different parameter settings

✅ Compared different strategy types

## Next Steps

1. **Strategy Comparison** - See notebook `02_strategy_comparison.ipynb` to compare multiple strategies side-by-side

2. **Parameter Tuning** - See notebook `03_parameter_tuning.ipynb` to systematically optimize parameters

3. **Results Analysis** - See notebook `04_results_analysis.ipynb` for advanced performance analytics

4. **Custom Strategies** - See `examples/custom_components_example.py` to create your own signals

## Questions?

- Check `docs/ADDING_CUSTOM_COMPONENTS.md` for extension guide
- Review `examples/yaml_strategy_example.py` for more usage patterns
- Read `README.md` for system overview

---

## Experiment on Your Own!

Try modifying the code above:

- Change the `instruments` list to include more contracts
- Adjust the `start_date` and `end_date` for a longer backtest
- Try different `risk_aversion` values
- Test different strategy templates from `list_templates()`

Have fun exploring! 🚀